In [ ]:
!pip install torch torchvision
!pip install numpy
!pip install opencv-python
!pip install albumentations
!pip install scikit-image
!pip install openslide-python
!apt-get install -y python3-openslide
!pip install openslide-python

In [1]:
from google.colab import files
uploaded = files.upload()

Saving fold1.zip to fold1.zip


In [2]:
!mkdir -p /content/pannuke/fold1
!mkdir -p /content/pannuke/fold2
!mkdir -p /content/pannuke/fold3
!unzip -q fold_1.zip -d /content/pannuke/fold1


unzip:  cannot find or open fold_1.zip, fold_1.zip.zip or fold_1.zip.ZIP.


SyntaxError: invalid character '├' (U+251C) (ipython-input-3026547097.py, line 2)

In [4]:
import os
data_dir = '/content/pannuke'
for fold in ['fold1', 'fold2', 'fold3']:
    fold_path = os.path.join(data_dir, fold)
    print(f"Contents of {fold_path}:")
    print(os.listdir(fold_path))


Contents of /content/pannuke/fold1:
[]
Contents of /content/pannuke/fold2:
[]
Contents of /content/pannuke/fold3:
[]


In [1]:
import numpy as np
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Define base directory
data_dir = '/content/pannuke'

# Load PanNuke Fold 1
def load_pannuke_fold(fold_name, data_dir=data_dir):
    images_path = os.path.join(data_dir, fold_name, 'images.npy')
    masks_path = os.path.join(data_dir, fold_name, 'masks.npy')
    for path in [images_path, masks_path]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
    images = np.load(images_path)  # Shape: (2654, 256, 256, 3)
    masks = np.load(masks_path)    # Shape: (2654, 256, 256, 6)
    # Convert to binary masks (nuclei vs. background)
    masks = np.sum(masks[:, :, :, :5], axis=-1) > 0  # Shape: (2654, 256, 256)
    masks = np.expand_dims(masks, axis=-1).astype(np.float32)  # Shape: (2654, 256, 256, 1)
    images = images.astype(np.float32)
    return images, masks

# Load Fold 1 and split into train/val/test
try:
    images, masks = load_pannuke_fold('fold1')
except FileNotFoundError as e:
    print(e)
    print("Please verify the dataset is in /content/pannuke/fold1.")
    exit()

# Split Fold 1: 60% train, 20% val, 20% test
train_images, temp_images, train_masks, temp_masks = train_test_split(
    images, masks, test_size=0.4, random_state=42
)
val_images, test_images, val_masks, test_masks = train_test_split(
    temp_images, temp_masks, test_size=0.5, random_state=42
)

print(f"Train images: {train_images.shape}, Train masks: {train_masks.shape}")
print(f"Val images: {val_images.shape}, Val masks: {val_masks.shape}")
print(f"Test images: {test_images.shape}, Test masks: {test_masks.shape}")

# Visualize a sample
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(train_images[0].astype(np.uint8))
plt.title("Image")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(train_images[0].astype(np.uint8))
plt.imshow(train_masks[0].squeeze(), cmap='Reds', alpha=0.4)
plt.title("Overlayed Mask")
plt.axis('off')
plt.tight_layout()
plt.show()

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_masks)).batch(16).shuffle(100).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_masks)).batch(16).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_masks)).batch(16).prefetch(tf.data.AUTOTUNE)

# Model (U-Net)
def conv_block(input_tensor, num_filters):
    x = tf.keras.layers.Conv2D(num_filters, (3, 3), padding='same', activation='relu')(input_tensor)
    x = tf.keras.layers.Conv2D(num_filters, (3, 3), padding='same', activation='relu')(x)
    return x

def decoder_block(input_tensor, skip_tensor, num_filters):
    x = tf.keras.layers.Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(input_tensor)
    x = tf.keras.layers.Concatenate()([x, skip_tensor])
    x = conv_block(x, num_filters)
    return x

def build_unet(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    c1 = conv_block(inputs, 64)
    p1 = tf.keras.layers.MaxPooling2D((2, 2))(c1)
    c2 = conv_block(p1, 128)
    p2 = tf.keras.layers.MaxPooling2D((2, 2))(c2)
    c3 = conv_block(p2, 256)
    p3 = tf.keras.layers.MaxPooling2D((2, 2))(c3)
    c4 = conv_block(p3, 512)
    p4 = tf.keras.layers.MaxPooling2D((2, 2))(c4)
    bn = conv_block(p4, 1024)
    d1 = decoder_block(bn, c4, 512)
    d2 = decoder_block(d1, c3, 256)
    d3 = decoder_block(d2, c2, 128)
    d4 = decoder_block(d3, c1, 64)
    outputs = tf.keras.layers.Conv2D(1, (1, 1), activation='sigmoid')(d4)
    return tf.keras.models.Model(inputs=inputs, outputs=outputs)

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2. * intersection + smooth) / (union + smooth)

# Build and train
model = build_unet((256, 256, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', dice_coef])

earlystop_cb = tf.keras.callbacks.EarlyStopping(
    patience=4, restore_best_weights=True, monitor='val_loss', mode='min', verbose=1
)
reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    factor=0.5, patience=3, monitor='val_loss', mode='min', verbose=1
)

history = model.fit(train_ds, epochs=30, validation_data=val_ds, callbacks=[earlystop_cb, reduce_lr_cb])

# Save model
model.save('/content/unet_model.keras')

# Plot Dice coefficient
plt.figure(figsize=(8, 5))
plt.plot(history.history['dice_coef'], label='Training Dice', linewidth=2)
plt.plot(history.history['val_dice_coef'], label='Validation Dice', linewidth=2)
plt.title('Dice Coefficient Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Dice Coefficient')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Evaluate on test set
val_imgs, val_masks = next(iter(test_ds))
preds = model.predict(val_imgs)
preds = (preds > 0.5).astype(np.uint8)

# Visualize results
n = 5
plt.figure(figsize=(15, n * 3))
for i in range(n):
    plt.subplot(n, 3, i*3 + 1)
    plt.imshow(val_imgs[i].numpy().astype("uint8"))
    plt.title("Image")
    plt.axis('off')
    plt.subplot(n, 3, i*3 + 2)
    plt.imshow(val_imgs[i].numpy().astype("uint8"))
    plt.imshow(val_masks[i].numpy().squeeze(), alpha=0.4, cmap='Reds')
    plt.title("Ground Truth Mask")
    plt.axis('off')
    plt.subplot(n, 3, i*3 + 3)
    plt.imshow(val_imgs[i].numpy().astype("uint8"))
    plt.imshow(preds[i].squeeze(), alpha=0.4, cmap='Blues')
    plt.title("Predicted Mask")
    plt.axis('off')
plt.tight_layout()
plt.show()

File not found: /content/pannuke/fold1/images.npy
Please verify the dataset is in /content/pannuke/fold1.


NameError: name 'images' is not defined